# Expanding on Gradient Descent

## Squared Errors

### Sum of the Squared Errors (SSE)

$$E = \frac{1}{2} \sum_{\mu} \sum_{j} \left[ y_j^{\mu} - \hat{y}_j^{\mu} \right]^2$$

This is the **squared error** loss function — an alternative to cross-entropy that measures how far the model's predictions are from the true values by squaring the difference.

| Symbol | Meaning |
|--------|---------|
| $E$ | Total error across all samples and output units |
| $\mu$ | Index over training samples |
| $j$ | Index over output units (neurons in the output layer) |
| $y_j^{\mu}$ | True value of output unit $j$ for sample $\mu$ |
| $\hat{y}_j^{\mu}$ | Model's predicted value of output unit $j$ for sample $\mu$ |
| $\frac{1}{2}$ | Convenience factor — makes the derivative cleaner (the 2 from the square cancels it) |

**How it works:** For each sample $\mu$ and each output $j$, compute the difference between what the model predicted and the true value, square it (so negative errors don't cancel positive ones), and sum everything up.

> **Real-life example:** Imagine you're predicting house prices. For house A, you predicted \$300k but it sold for \$350k — that's an error of \$50k, squared = 2,500,000,000. For house B, you predicted \$200k and it sold for \$205k — error of \$5k, squared = 25,000,000. The squaring means the big miss on house A dominates the total error, which pushes the model to fix its worst predictions first. The $\frac{1}{2}$ doesn't affect which direction the model improves — it just makes the math cleaner when we take derivatives.

## Gradient Descent Weight Update

### Weight Update Rule

$$\Delta w_i = \eta \, \delta \, x_i$$

| Symbol | Meaning |
|--------|---------|
| $\Delta w_i$ | The change to apply to weight $w_i$ |
| $\eta$ | Learning rate — controls how big each update step is |
| $\delta$ | Error term — how much this neuron is "responsible" for the error |
| $x_i$ | The $i$-th input feature |

**How it works:** Each weight gets nudged by an amount proportional to (1) how large the error was, (2) how large the input was, and (3) how bold you want to be with updates (the learning rate).

> **Real-life example:** You're adjusting the bass knob on a speaker. If the sound is way off ($\delta$ is large) and the bass is already very loud ($x_i$ is large), you make a bigger adjustment. If the sound is barely off or the bass is nearly silent, you barely touch it. The learning rate $\eta$ is how cautious you are — a bold engineer turns the knob a lot, a cautious one makes tiny moves.

---

### The Error Term $\delta$

$$\delta = (y - \hat{y}) \, f'(h) = (y - \hat{y}) \, f'\!\left(\sum_i w_i x_i\right)$$

| Symbol | Meaning |
|--------|---------|
| $\delta$ | Error term for this neuron |
| $y$ | True label |
| $\hat{y}$ | Model's predicted output |
| $(y - \hat{y})$ | Raw prediction error — how wrong the model was |
| $f'(\cdot)$ | Derivative of the activation function $f$ |
| $h = \sum_i w_i x_i$ | The weighted sum of inputs before activation |

**How it works:** The error term combines two things — *how wrong* the prediction was $(y - \hat{y})$, and *how sensitive* the activation function was at that point $f'(h)$. If the activation function is nearly flat (derivative ≈ 0), even a large prediction error produces a tiny weight update — the neuron was "saturated" and hard to move.

**Connection to the weight update:** This is why the gradient descent update is $\Delta w_i = \eta \delta x_i$ and not just $\eta (y - \hat{y}) x_i$ — the activation function's slope is baked into $\delta$.

> **Real-life example:** Think of a volume knob that's stuck near maximum. Even if the music is too loud (large error), turning the knob barely changes the volume (small $f'(h)$) — so the correction is small. Conversely, when the knob is in the sensitive middle range, even a small turn makes a big difference, so the correction is larger. The activation derivative $f'(h)$ captures exactly this "how responsive is this neuron right now?" factor.

### Gradient Descent - The Code

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 /  (1 + np.exp(-x))

def sigmoid_prime(x):
    return sigmoid(x) * (1 - sigmoid(x))

x = np.array([0.1,0.3])
y = 0.2

weights = np.array([-0.8, 0.5])

learnrate = 0.5

h = x[0]*weights[0] + x[1]*weights[1]
# or
# h = np.dot(x, weights)

nn_output = sigmoid(h)

error = y - nn_output

output_grad = sigmoid_prime(h)

error_term = error * output_grad

del_w = [
    learnrate * error_term * x[0],
    learnrate * error_term * x[1],
]
# or
# del_w = learnrate * error_term * x

## Derivative of the Sigmoid Function

The sigmoid function is defined as:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

Its derivative has a remarkably elegant form:

$$
\frac{d}{dx}\sigma(x) = \sigma(x) \cdot (1 - \sigma(x))
$$

| Symbol | Meaning |
|--------|---------|
| $\sigma(x)$ | The sigmoid function output (a value between 0 and 1) |
| $e^{-x}$ | The exponential decay term |
| $\frac{d}{dx}$ | Derivative with respect to $x$ |

### How it works

The derivative tells us how fast the sigmoid output changes as we adjust the input. The formula $\sigma(x)(1 - \sigma(x))$ means the slope depends entirely on the current output value — no need to recompute the exponential.

> **Real-life example:** Imagine a dimmer switch that smoothly transitions a light from off to on. The derivative tells you how *responsive* the dimmer is at any position. Near the middle (σ ≈ 0.5), small turns produce noticeable brightness changes (high derivative ≈ 0.25). At the extremes (σ ≈ 0 or 1), the light is nearly fully off or on, so turning the dial barely changes anything (low derivative ≈ 0).

### Derivation

Starting from the definition:

$$
\frac{d}{dx}\sigma(x) = \frac{d}{dx}\left[\frac{1}{1 + e^{-x}}\right]
$$

Rewrite as a power and apply the chain rule:

$$
= \frac{d}{dx}(1 + e^{-x})^{-1} = -(1 + e^{-x})^{-2} \cdot (-e^{-x})
$$

Simplify the double negative:

$$
= \frac{e^{-x}}{(1 + e^{-x})^2}
$$

Now factor this into a product of two fractions:

$$
= \frac{1}{1 + e^{-x}} \cdot \frac{e^{-x}}{1 + e^{-x}}
$$

Recognize that the first fraction is $\sigma(x)$. For the second fraction, add and subtract 1 in the numerator:

$$
= \sigma(x) \cdot \frac{(1 + e^{-x}) - 1}{1 + e^{-x}} = \sigma(x) \cdot \left(1 - \frac{1}{1 + e^{-x}}\right)
$$

This gives us the final result:

$$
= \sigma(x) \cdot (1 - \sigma(x))
$$

### Observations

- **Computational efficiency:** Once you compute $\sigma(x)$ during the forward pass, the derivative comes "for free" — just multiply `sigmoid_output * (1 - sigmoid_output)`. No exponentials needed in backpropagation.
- **Maximum slope at the midpoint:** The derivative $\sigma(1-\sigma)$ is maximized when $\sigma = 0.5$, giving a peak value of $0.25$. This is why gradients flow best through neurons with activations near $0.5$.
- **Vanishing gradient problem:** When $\sigma \approx 0$ or $\sigma \approx 1$, the derivative approaches $0$. In deep networks, this causes gradients to shrink exponentially across layers — the infamous "vanishing gradient" problem that motivated alternatives like ReLU.

## Mean Squared Error (MSE)

$$
E = \frac{1}{2m} \sum_{\mu} \left( y^{\mu} - \hat{y}^{\mu} \right)^2
$$

| Symbol | Meaning |
|--------|---------|
| $E$ | The mean squared error — average loss across all samples |
| $m$ | Total number of training samples |
| $\mu$ | Index over training samples (from 1 to $m$) |
| $y^{\mu}$ | True value for sample $\mu$ |
| $\hat{y}^{\mu}$ | Model's predicted value for sample $\mu$ |
| $\frac{1}{2}$ | Convenience factor — cancels the 2 from the derivative of the square |

### How it works

For each sample, compute the difference between the true value and the prediction, square it, then average over all $m$ samples. The $\frac{1}{2}$ doesn't change the optimal weights — it just makes the gradient $\frac{\partial E}{\partial w}$ cleaner by canceling the 2 that comes from differentiating the square.

> **Real-life example:** You're a weather forecaster and you predicted temperatures for 100 days. For each day, you compute (actual temp − predicted temp)², sum them all up, and divide by 100 (times ½). A single 20°F miss contributes 400 to your error, while ten 2°F misses only contribute 40 total. MSE punishes big mistakes harshly, encouraging your model to avoid catastrophic errors even at the cost of small ones.

### Relationship to SSE

This is the **normalized** version of the Sum of Squared Errors (SSE) from earlier in this notebook:

$$
\text{SSE} = \frac{1}{2} \sum_{\mu} \sum_{j} \left( y_j^{\mu} - \hat{y}_j^{\mu} \right)^2
\quad \Rightarrow \quad
\text{MSE} = \frac{\text{SSE}}{m}
$$

- **SSE** sums over all samples and all output units — it grows with dataset size
- **MSE** divides by $m$ to get a per-sample average — it stays comparable across different dataset sizes

### Observations

- **Scale-independent comparison:** Because MSE averages over samples, you can compare models trained on different-sized datasets. An MSE of 0.05 means the same thing whether you have 100 or 10,000 samples.
- **Gradient scales with batch size:** When computing gradients for mini-batch gradient descent, the $\frac{1}{m}$ factor ensures that the gradient magnitude doesn't explode with larger batches — you're always working with the *average* gradient.
- **Units are squared:** If your target is in dollars, MSE is in dollars². To get back to interpretable units, take the square root → **RMSE** (Root Mean Squared Error).

## The Complete Gradient Descent Algorithm

Here's the general algorithm for updating the weights with gradient descent:

1. **Initialize weight steps to zero:**
$$\Delta w_i = 0$$

2. **For each record in the training data:**
   - Make a forward pass through the network, calculating the output:
   $$\hat{y} = f\left(\sum_i w_i x_i\right)$$
   - Calculate the error term for the output unit:
   $$\delta = (y - \hat{y}) \cdot f'\left(\sum_i w_i x_i\right)$$
   - Accumulate the weight step:
   $$\Delta w_i = \Delta w_i + \delta \, x_i$$

3. **Update the weights** (after processing all records):
$$w_i = w_i + \eta \frac{\Delta w_i}{m}$$

4. **Repeat for $e$ epochs.**

| Symbol | Meaning |
|--------|---------|
| $\Delta w_i$ | Accumulated weight change for weight $i$ |
| $w_i$ | The $i$-th weight |
| $x_i$ | The $i$-th input feature |
| $\hat{y}$ | Model's predicted output |
| $y$ | True label |
| $f(\cdot)$ | Activation function (e.g., sigmoid) |
| $f'(\cdot)$ | Derivative of the activation function |
| $\delta$ | Error term — combines prediction error with activation sensitivity |
| $\eta$ | Learning rate — controls step size |
| $m$ | Number of training records |
| $e$ | Number of epochs |

### How it works

Each epoch processes the entire dataset. For each sample, we compute how wrong the prediction was and accumulate the suggested weight changes. After seeing all samples, we apply the **average** of those changes to the weights. This averaging (dividing by $m$) smooths out noise from individual samples — a single outlier won't throw off the whole update.

> **Real-life example:** Imagine you're a chef adjusting a recipe. You serve the dish to 100 people and each gives feedback: "needs more salt," "too salty," etc. Instead of reacting to each comment one by one (which could make you over-correct), you collect *all* the feedback, average it, and make one measured adjustment. Then you repeat the process (another epoch) until the dish is just right. The learning rate $\eta$ is how bold you are with each adjustment — too high and you overcorrect, too low and it takes forever to converge.

### Observations

- **Batch vs. Stochastic:** This describes **batch gradient descent** — you accumulate $\Delta w_i$ over all $m$ samples before updating. In **stochastic gradient descent (SGD)**, you update after each sample (no accumulation, $m=1$). **Mini-batch** is in between — accumulate over a small batch, then update.
- **Why average?** Dividing by $m$ keeps the gradient magnitude consistent regardless of dataset size. Without it, larger datasets would produce larger updates, making the learning rate harder to tune.
- **Epochs vs. iterations:** One epoch = one pass through the entire dataset. With batch GD, one epoch = one weight update. With SGD, one epoch = $m$ weight updates.

## Implementing with NumPy

Initializing the weights

In [ ]:
weights = np.random.normal(scale=1/n_features**.5, size=n_features)

Using  `np.dot()` to calculate the product of two arrays (calculate <i>h</i> for us).

For two 1D arrays, the dot product multiplies corresponding elements and then sums the results.

In [ ]:
output_in = np.dot(weights, inputs)

### Efficiency tip!

You can save some calculations since we're using a sigmoid here. For the sigmoid function:

$$f'(h) = f(h) \cdot (1 - f(h))$$

That means once you calculate $f(h)$ — the activation of the output unit — you can reuse it to calculate the gradient for the error term. No need to call the derivative function separately!

**In code terms:**
```python
# Instead of computing sigmoid_prime(h) from scratch...
output = sigmoid(h)           # f(h) — you need this anyway
gradient = output * (1 - output)  # f'(h) — reuse output!
```

> This is why the code in the earlier cell uses `sigmoid_prime(h)` which internally calls `sigmoid(h)` — but in optimized implementations, you'd compute `sigmoid(h)` once and derive both the output and its gradient from that single value.